## Импорты

In [1]:
CONFIG_NAME = "custom_lstm_correction.yaml"
#CONFIG_NAME = "custom_CNN_RNN.yaml"

In [2]:
import sys
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)

In [3]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [4]:
CONFIG_PATH = f"acoustic/configs/{CONFIG_NAME}"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [5]:
dataset = load_and_prepare_dataset(cfg)

Applying filters: 0it [00:00, ?it/s]


## Инициализация модели

In [6]:
model, processor, data_collator = build_model(cfg)

print("Model built")

Model built


## Создание и загрузка метрик, callbacks, trainer

In [7]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

Metrics: ['wer', 'cer', 'f1', 'detailed_stats', 'ser', 'space_wer']


In [8]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [9]:
print("Starting training")
trainer.train()

Starting training


Training: 100%|██████████| 148410/148410 [00:00<?, ?step/s]

{'train_runtime': 0.2412, 'train_samples_per_second': 9846111.577, 'train_steps_per_second': 615391.044, 'train_loss': 0.0, 'epoch': 5.0}


## Проверка

In [10]:
print("Demo on validation examples")
import random
import torch
from acoustic.models import get_generate_method

eval_dataset = dataset['validation']

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(10, len(eval_dataset)))
    device = next(model.parameters()).device
    model.eval()
    builder_key = cfg['model']['builder']
    generate_fn = get_generate_method(builder_key)

    
    for i in indices:
        example = eval_dataset[i]

        # Ветка для текстового корректора
        if builder_key == "correction_model":
            stt_text = "исправь: " + example["stt_text"]
            ref_text = example["reference"]

            inputs = processor(stt_text, return_tensors="pt", truncation=True, padding=True, max_length=128)
            input_data = inputs["input_ids"].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" ASR output : {stt_text}")
            print(f" Corrected  : {pred_text}")
            print(f" Reference  : {ref_text}")

        elif builder_key == "lstm_correction":
            stt_text = example["stt_text"]
            ref_text = example["reference"]

            inputs = processor(stt_text, return_tensors="pt", truncation=True, padding=True, max_length=220)
            input_data = inputs["input_ids"].to(device)

            with torch.no_grad():
                outputs = model(input_data)          # возвращает {"logits": ...}
            pred_ids = outputs["logits"]
            pred_text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" ASR output : {stt_text}")
            print(f" Corrected  : {pred_text}")
            print(f" Reference  : {ref_text}")

        else:
            audio_array = example["audio"]["array"]
            ref_text = example["sentence"]

            inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
            input_key = "input_features" if "input_features" in inputs else "input_values"
            input_data = inputs[input_key].to(device)

            with torch.no_grad():
                predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            print(f"\nExample {i+1}:")
            print(f" Reference: {ref_text}")
            print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")

Demo on validation examples

Example 913:
 ASR output : джой покажи мне воланс карте визя
 Corrected  : джой покажи мне валовск карты виза
 Reference  : джой покажи мне баланс карты виза

Example 205:
 ASR output : прахаждени видьма дика хото
 Corrected  : прохождение видео макко холод
 Reference  : прохождение ведьмак дикая охота

Example 2254:
 ASR output : поставь серия двенадцать последнигой сезона одабрасы
 Corrected  : поставь серию двенадцать последнего сезона отбросы
 Reference  : поставь серию двенадцать последнего сезона отбросы

Example 2007:
 ASR output : лучшие семеный сериал ло прошной год осмотреть
 Corrected  : лучшие семейные сериалы прошлогодный смотреть
 Reference  : лучший семейный сериал за прошлый год посмотреть

Example 1829:
 ASR output : какава стоимость обме на канадского доллара франке
 Corrected  : какова стоимость обмена канадского доллара в франки
 Reference  : какова стоимость обмена канадского доллара в франки

Example 1144:
 ASR output : найти на смотре